# 03e · Ținta: pierderea observată în 2024
Fiecare parcelă e comparată doar cu anii în care a avut același tip de cultură. Tipul se citește din NDVI-ul din aprilie: verde = cultură de toamnă, gol = cultură de primăvară.

- Rezultat: `data/out/parcel_target.parquet` (`is_arable`, `crop_group_2024`, `loss_source`, `ref`, `loss_obs`, `loss_pct`, `n_ref_years`).
- Sursa e HR-VPP `TPROD` dacă a fost descărcat (acoperă și culturile de toamnă); altfel NDVI iulie–august, doar pentru culturile de primăvară.
- Parcelele non-arabile rămân în tabel cu `is_arable = False` și nu intră nici în model, nici în colorarea hărții.

In [1]:
import os, sys
if not os.path.exists('gisday_config.py') and os.path.exists('../gisday_config.py'):
    os.chdir('..')
sys.path.insert(0, os.getcwd())
from gisday_config import *
from gisday_lib import crop_group, like_for_like_loss, crop_group_agreement
import numpy as np, pandas as pd
F = pd.read_parquet(OUT / 'parcel_features.parquet')
print(F.shape)

(17466, 67)


In [2]:
codes = F['crop'].round()
F['is_arable'] = codes.between(211, 290)        # clasele de culturi din EUCROPMAP (verificate în proprietățile imaginii)
for y in BASE_YEARS:
    F[f'grp_{y}'] = crop_group(F[f'ndvi_apr_{y}'])
F[f'grp_{YEAR}'] = crop_group(F['ndvi_04'])
m_incert = F['ndvi_04'].between(0.38, 0.45) | (codes == 250)
F.loc[m_incert, f'grp_{YEAR}'] = 'grup_incert'
n_ar = int(F.is_arable.sum())
n_inc = int((F.is_arable & m_incert).sum())
print(f'parcele arabile: {n_ar} din {len(F)} | grup_incert: {n_inc} ({n_inc/n_ar:.2%})')
print(F.loc[F.is_arable, f'grp_{YEAR}'].value_counts().to_string())

parcele arabile: 13237 din 17466 | grup_incert: 1036 (7.83%)
grp_2024
toamna         6365
primavara      5836
grup_incert    1036


In [3]:
# Control: tipul citit din aprilie 2022 trebuie să se potrivească cu sezonul culturii din EUCROPMAP 2022
share, n = crop_group_agreement(F.loc[F.is_arable, 'grp_2022'], codes[F.is_arable])
print(f'acord cu EUCROPMAP 2022: {share:.1%} din {n} parcele')
assert share >= 0.85, 'Pragurile NDVI din aprilie nu separă bine culturile: oprire, vezi etapa E3 din plan'

acord cu EUCROPMAP 2022: 93.7% din 12809 parcele


In [4]:
ref_cols = [f'tprod_{y}' for y in BASE_YEARS]
n_ani_baza = F[ref_cols].notna().sum(axis=1)
tprod_mean = F[ref_cols].mean(axis=1)
tprod_std = F[ref_cols].std(axis=1, ddof=1)
valid_z = (n_ani_baza >= 5) & F[f'tprod_{YEAR}'].notna() & (tprod_std > 0)
z_tprod = np.where(valid_z, (F[f'tprod_{YEAR}'] - tprod_mean) / tprod_std, np.nan)

L = like_for_like_loss(F, 'tprod_{y}', 'grp_{y}', YEAR, BASE_YEARS)
loss_rel_arable = L['loss_rel'].where(F['is_arable'])
p01, p99 = loss_rel_arable.quantile([0.01, 0.99])
n_clipped = int(((loss_rel_arable < p01) | (loss_rel_arable > p99)).sum())
loss_rel_clipped = loss_rel_arable.clip(p01, p99)

T = pd.DataFrame({
    'parcel_id': F['parcel_id'],
    'is_arable': F['is_arable'],
    'tip_cultura_2024': F[f'grp_{YEAR}'],
    'crop_group_2024': F[f'grp_{YEAR}'],
    'ref': L['ref'],
    'n_ref_years': L['n_ref_years'],
    'loss_rel_raw': loss_rel_arable,
    'loss_rel': loss_rel_clipped,
    'loss': loss_rel_clipped,
    'loss_obs': loss_rel_clipped,
    'loss_pct': L['loss_pct'],
    'n_ani_baza': n_ani_baza,
    'tprod_mean_2017_2023': tprod_mean,
    'tprod_std_2017_2023': tprod_std,
    'z_tprod_2024': z_tprod,
    'loss_source': 'hrvpp_tprod_lfl_rel',
})
assert T.parcel_id.is_unique
T.to_parquet(OUT / 'parcel_target.parquet')
ok = T[T.is_arable & T.loss.notna()]
print(f'sursa: {T.loss_source.iloc[0]} | parcele arabile cu țintă (>=2 ani ref): {len(ok)} | fără țintă (<2 ani ref): {int((T.is_arable & T.loss.isna()).sum())}')
print(f'tăiere P1/P99: P1={p01:.4f}, P99={p99:.4f} | parcele afectate: {n_clipped}')
print(ok.groupby('tip_cultura_2024')['loss'].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(4).to_string())

sursa: hrvpp_tprod_lfl_rel | parcele arabile cu țintă (>=2 ani ref): 11048 | fără țintă (<2 ani ref): 2189
tăiere P1/P99: P1=-1.7747, P99=0.7565 | parcele afectate: 222
                   count    mean     std     min      1%      5%     25%     50%     75%     95%     99%     max
tip_cultura_2024                                                                                                
primavara         4973.0  0.2063  0.4418 -1.7747 -1.5612 -0.6295  0.0143  0.3010  0.5208  0.7111  0.7565  0.7565
toamna            6075.0 -0.0979  0.4867 -1.7747 -1.7747 -1.0826 -0.3316 -0.0017  0.2479  0.5109  0.6436  0.7565
